In [19]:
!pip install rasterio shapely
!pip install torch torchgeo
# Cell 0: Test TorchGeo Approach for Reading GeoTIFF Metadata
import os
import torch
from torchgeo.datasets import RasterDataset
from torch.utils.data import DataLoader
from torchgeo.samplers import GridGeoSampler
from torchgeo.datasets import stack_samples, unbind_samples
from rasterio.transform import from_bounds

  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached antlr4_python3_runtime-4.9.3-py3-none-any.whl
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.11.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 605.0/605.0 kB 11.0 MB/s eta 0:00:00
Using cached einops-0.8.1-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━

# TIF Zone Classification - Direct GeoTIFF Approach
This notebook classifies 400x400 GeoTIFF images into agroclimatic zones using direct geographic metadata instead of hash matching.

In [20]:
# Define custom dataset class
class TestTiffDataset(RasterDataset):
    filename_glob = "*.tif*"
    filename_regex = r"()"
    date_format = "%Y%m%d"
    is_image = True
    separate_files = True
    all_bands = ["red", "green", "blue"]
    rgb_bands = ["red", "green", "blue"]
    
    def __getitem__(self, index):
        sample_dict = super().__getitem__(index)
        
        # Compute the affine transform for this image
        image = sample_dict['image']
        transform = self.get_transform(sample_dict["bounds"], image.shape[1], image.shape[2])
        sample_dict["transform"] = transform
        return sample_dict
    
    def get_transform(self, bounds, height, width):
        # Convert a BoundingBox in pixel coordinates to an affine transform
        left, top, right, bottom = bounds.minx, bounds.maxy, bounds.maxx, bounds.miny
        return from_bounds(left, bottom, right, top, width, height)

# Test directory containing your GeoTIFF
# UPDATE THIS PATH to point to directory containing your test file
test_dir = './test_file'

print("Loading dataset with TorchGeo...")
dataset = TestTiffDataset(test_dir)
print(f"Dataset loaded: {dataset}")
print(f"Number of files found: {len(dataset)}")
print(f"Bounding box: {dataset.bounds}")

# Load one sample to check metadata
if len(dataset) > 0:
    sampler = GridGeoSampler(dataset, size=400, stride=400)
    dataloader = DataLoader(dataset, sampler=sampler, collate_fn=stack_samples)
    
    # Get first batch
    for batch in dataloader:
        unbinded = unbind_samples(batch)
        sample = unbinded[0]
        
        print(f"\nSample metadata:")
        print(f"  Image shape: {sample['image'].shape}")
        print(f"  Bounds: {sample['bounds']}")
        print(f"  Transform: {sample['transform']}")
        
        # Calculate center point
        bds = sample['bounds']
        center_x = (bds.minx + bds.maxx) / 2
        center_y = (bds.miny + bds.maxy) / 2
        print(f"  Center point: ({center_x}, {center_y})")
        
        # Check if bounds are real geographic coordinates or just pixel coordinates
        if bds.minx == 0 and bds.miny == 0:
            print("\n⚠️  WARNING: Bounds appear to be pixel coordinates, not geographic coordinates")
            print("   This means the GeoTIFF lacks proper georeferencing")
        else:
            print("\n✓ Bounds appear to be real geographic coordinates")
            print("  Ready for zone classification!")
        
        break
else:
    print("No GeoTIFF files found in the directory")

Loading dataset with TorchGeo...


/opt/miniconda3/lib/python3.13/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


CPLE_AppDefinedError: Unable to compute a transformation between pixel/line and georeferenced coordinates for ./test_file/gpugeomachine3_DATASET_rajasthanzonedataset_Images_013REXC4PT_chipid319.tiff. There is no affine transformation and no GCPs. Specify transformation option SRC_METHOD=NO_GEOTRANSFORM to bypass this check.

In [ ]:
# Cell 1: Imports
import os
import geopandas as gpd
import rasterio
from shapely import Point
import pandas as pd

In [ ]:
# Cell 2: Geographic Intersection Function
def get_region(shape, key, point):
    """Find which polygon in shapefile contains the given point."""
    for row_id, row in shape.iterrows():
        if row['geometry'].intersects(point):
            return row[key]
    return 'UNKNOWN'

In [ ]:
# Cell 3: Karnataka Setup
# Load Karnataka district boundaries
shp_file = './karnataka_footprints/DISTRICT_BOUNDARY.shp'
karnataka_data = gpd.read_file(shp_file).to_crs(32643)
karnataka_data = karnataka_data[karnataka_data['STATE'] == 'KARNATAKA']

print(f"Loaded {len(karnataka_data)} Karnataka districts")
print(f"Districts: {sorted(karnataka_data['District'].unique())}")

In [ ]:
# Cell 4: Karnataka District to Zone Mapping
karnataka_district_to_zone = {
    'SHIVAMOGGA': 'SOUTHERN TRANSITION', 
    'UTTARA  KANNADA': 'HILL', 
    'DAVANGERE': 'CENTRAL DRY', 
    'CHITRADURGA': 'CENTRAL DRY', 
    'BALLARI': 'NORTH EAST DRY', 
    'DHARWAD': 'WESTERN TRANSITION', 
    'GADAG': 'NORTHERN DRY', 
    'KOPPAL': 'NORTH EAST DRY', 
    'RAICHUR': 'NORTH EAST DRY', 
    'YADGIR': 'NORTH EAST DRY',
    'KODAGU': 'SOUTHERN DRY', 
    'MANDYA': 'SOUTHERN DRY', 
    'RAMANAGARAM': 'EASTERN DRY', 
    'DAKSHINA  KANNADA': 'COASTAL', 
    'HASSAN': 'SOUTHERN TRANSITION', 
    'KOLAR': 'EASTERN DRY', 
    'BENGALURU RURAL': 'EASTERN DRY', 
    'UDUPI': 'COASTAL', 
    'TUMAKURU': 'CENTRAL DRY', 
    'CHIKKABALLAPURA': 'EASTERN DRY'
}

print(f"Mapped {len(karnataka_district_to_zone)} districts to zones")

In [ ]:
# Cell 5: Process Karnataka GeoTIFFs
# UPDATE THESE PATHS:
karnataka_input_dirs = [
    # Add paths to your Karnataka GeoTIFF directories
    # Example: './path/to/KarnatakaTrees0/images/default/',
]

output_csv = 'karnataka_image_zones.csv'

with open(output_csv, 'w') as f:
    f.write('filename,zone\n')
    
    total = 0
    classified = 0
    unknown = 0
    zone_counts = {}
    
    for img_dir in karnataka_input_dirs:
        if not os.path.exists(img_dir):
            print(f"Directory not found: {img_dir}")
            continue
            
        print(f"Processing: {img_dir}")
        
        for filename in os.listdir(img_dir):
            if not filename.endswith('.tif') and not filename.endswith('.tiff'):
                continue
                
            filepath = os.path.join(img_dir, filename)
            total += 1
            
            try:
                with rasterio.open(filepath) as src:
                    # Get center point from bounds
                    bounds = src.bounds
                    center_lon = (bounds.left + bounds.right) / 2
                    center_lat = (bounds.bottom + bounds.top) / 2
                    
                    # Transform to shapefile CRS if needed
                    from pyproj import Transformer
                    transformer = Transformer.from_crs(src.crs, karnataka_data.crs, always_xy=True)
                    x, y = transformer.transform(center_lon, center_lat)
                    pt = Point(x, y)
                    
                    # Find district
                    district = get_region(karnataka_data, 'District', pt)
                    
                    # Map to zone
                    if district in karnataka_district_to_zone:
                        zone = karnataka_district_to_zone[district]
                        classified += 1
                    else:
                        zone = 'UNKNOWN'
                        unknown += 1
                    
                    zone_counts[zone] = zone_counts.get(zone, 0) + 1
                    f.write(f'{filename},{zone}\n')
                    
            except Exception as e:
                print(f"Error processing {filename}: {e}")
                f.write(f'{filename},ERROR\n')
    
    print(f"\nTotal images: {total}")
    print(f"Successfully classified: {classified}")
    print(f"Unknown: {unknown}")
    print(f"\nZone distribution:")
    for zone, count in sorted(zone_counts.items()):
        print(f"  {zone}: {count}")
    print(f"\nOutput saved to: {output_csv}")

In [ ]:
# Cell 6: Rajasthan Setup
# Load Rajasthan district boundaries
raj_shp_file = './rajasthan_districts/districts_of_karnataka.shp'  # Note: misnamed but contains Rajasthan data
rajasthan_data = gpd.read_file(raj_shp_file).to_crs(32643)

print(f"Loaded {len(rajasthan_data)} Rajasthan districts")
print(f"Districts: {sorted(rajasthan_data['ADM2_NAME'].unique())}")

In [ ]:
# Cell 7: Rajasthan District to Zone Mapping
rajasthan_district_to_zone = {
    'Alwar': 'FLOOD PRONE EASTERN',
    'Bharatpur': 'FLOOD PRONE EASTERN',
    'Dhaulpur': 'FLOOD PRONE EASTERN',
    'Karauli': 'FLOOD PRONE EASTERN',
    'Sawai Madhopur': 'FLOOD PRONE EASTERN',
    'Kota': 'HUMID SOUTHEASTERN',
    'Jhalawar': 'HUMID SOUTHEASTERN',
    'Bundi': 'HUMID SOUTHEASTERN',
    'Baran': 'HUMID SOUTHEASTERN',
    'Barmer': 'HYPERARID WESTERN',
    'Bikaner': 'HYPERARID WESTERN',
    'Jaisalmer': 'HYPERARID WESTERN',
    'Sriganganagar': 'IRRIGATED NORTHERN',
    'Hanumangarh': 'IRRIGATED NORTHERN'
}

print(f"Mapped {len(rajasthan_district_to_zone)} districts to zones")

In [ ]:
# Cell 8: Process Rajasthan GeoTIFFs
# UPDATE THESE PATHS:
rajasthan_input_dirs = [
    # Add paths to your Rajasthan GeoTIFF directories
    # Example: './path/to/RajasthanTrees0/images/default/',
]

output_csv = 'rajasthan_image_zones.csv'

with open(output_csv, 'w') as f:
    f.write('filename,zone\n')
    
    total = 0
    classified = 0
    unknown = 0
    zone_counts = {}
    
    for img_dir in rajasthan_input_dirs:
        if not os.path.exists(img_dir):
            print(f"Directory not found: {img_dir}")
            continue
            
        print(f"Processing: {img_dir}")
        
        for filename in os.listdir(img_dir):
            if not filename.endswith('.tif') and not filename.endswith('.tiff'):
                continue
                
            filepath = os.path.join(img_dir, filename)
            total += 1
            
            try:
                with rasterio.open(filepath) as src:
                    # Get center point from bounds
                    bounds = src.bounds
                    center_lon = (bounds.left + bounds.right) / 2
                    center_lat = (bounds.bottom + bounds.top) / 2
                    
                    # Transform to shapefile CRS if needed
                    from pyproj import Transformer
                    transformer = Transformer.from_crs(src.crs, rajasthan_data.crs, always_xy=True)
                    x, y = transformer.transform(center_lon, center_lat)
                    pt = Point(x, y)
                    
                    # Find district
                    district = get_region(rajasthan_data, 'ADM2_NAME', pt)
                    
                    # Map to zone
                    if district in rajasthan_district_to_zone:
                        zone = rajasthan_district_to_zone[district]
                        classified += 1
                    else:
                        zone = 'UNKNOWN'
                        unknown += 1
                    
                    zone_counts[zone] = zone_counts.get(zone, 0) + 1
                    f.write(f'{filename},{zone}\n')
                    
            except Exception as e:
                print(f"Error processing {filename}: {e}")
                f.write(f'{filename},ERROR\n')
    
    print(f"\nTotal images: {total}")
    print(f"Successfully classified: {classified}")
    print(f"Unknown: {unknown}")
    print(f"\nZone distribution:")
    for zone, count in sorted(zone_counts.items()):
        print(f"  {zone}: {count}")
    print(f"\nOutput saved to: {output_csv}")

In [ ]:
# Cell 9: California Setup (Optional)
# Load California state boundaries and USDA zones
cali_shp = './ca_state/CA_State.shp'
cali_borders = gpd.read_file(cali_shp).to_crs(4269)

usda_shp = './phzm_us_zones_shp_2023/phzm_us_zones_shp_2023.shp'
usda_data = gpd.read_file(usda_shp)

# Intersect USDA zones with California
california_zones = usda_data.overlay(cali_borders).to_crs(4326)

print(f"Loaded {len(california_zones)} California hardiness zones")
print(f"Zones: {sorted(california_zones['zone'].unique())}")

In [ ]:
# Cell 10: Process California GeoTIFFs (Optional)
# UPDATE THESE PATHS:
california_input_dirs = [
    # Add paths to your California GeoTIFF directories
    # Example: './path/to/CaliforniaTrees0/images/default/',
]

output_csv = 'california_image_zones.csv'

with open(output_csv, 'w') as f:
    f.write('filename,zone\n')
    
    total = 0
    classified = 0
    unknown = 0
    zone_counts = {}
    
    for img_dir in california_input_dirs:
        if not os.path.exists(img_dir):
            print(f"Directory not found: {img_dir}")
            continue
            
        print(f"Processing: {img_dir}")
        
        for filename in os.listdir(img_dir):
            if not filename.endswith('.tif') and not filename.endswith('.tiff'):
                continue
                
            filepath = os.path.join(img_dir, filename)
            total += 1
            
            try:
                with rasterio.open(filepath) as src:
                    # Get center point from bounds
                    bounds = src.bounds
                    center_lon = (bounds.left + bounds.right) / 2
                    center_lat = (bounds.bottom + bounds.top) / 2
                    
                    # Transform to shapefile CRS if needed
                    from pyproj import Transformer
                    transformer = Transformer.from_crs(src.crs, california_zones.crs, always_xy=True)
                    x, y = transformer.transform(center_lon, center_lat)
                    pt = Point(x, y)
                    
                    # Find zone directly
                    zone = get_region(california_zones, 'zone', pt)
                    
                    if zone != 'UNKNOWN':
                        classified += 1
                    else:
                        unknown += 1
                    
                    zone_counts[zone] = zone_counts.get(zone, 0) + 1
                    f.write(f'{filename},{zone}\n')
                    
            except Exception as e:
                print(f"Error processing {filename}: {e}")
                f.write(f'{filename},ERROR\n')
    
    print(f"\nTotal images: {total}")
    print(f"Successfully classified: {classified}")
    print(f"Unknown: {unknown}")
    print(f"\nZone distribution:")
    for zone, count in sorted(zone_counts.items()):
        print(f"  {zone}: {count}")
    print(f"\nOutput saved to: {output_csv}")